# Spine preprocessing

Pipeline for the neuron-model `S` module (`docs/neuron-model/s-module-preprocessing.md`):

1. seal meshes + attachment loop / cap
2. false spine detection
3. local coordinate frame (`orient_spines_w_holes`)
4. logical branch merge + `metadata.json`/`manifest.parquet` (aggregates every other stage, runs last)
5. geometric QC of `local_sealed` meshes
6. point clouds 2048/4096/8192 x 4 seeds
7. SDF samples
8. morphometrics (`src/spine_analysis`, on `local_sealed_mesh`)
9. metadata (species/health/cell_type/cell_type_binary/compartment)

In [ ]:
%load_ext autoreload
%autoreload 2
# Reloads modules under src/ automatically before running each cell, so
# edits to .py files show up without restarting the kernel. Caveats: it
# won't pick up changes to a dataclass's fields for objects already
# constructed (e.g. an existing `cfg`), or `from x import y` bindings
# captured before the edit (re-run the import cell after such changes) -
# see https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html

from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Repository root was not found.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.neuron_model import (
    SpinesInfoConfig,
    SpinePreprocessingConfig,
    build_spines_info,
    discover_spines,
    run_false_spine_detection_stage,
    run_full_spine_preprocessing_pipeline,
    run_manifest_stage,
    run_mesh_qc_stage,
    run_orientation_stage,
    run_pointcloud_stage,
    run_sdf_stage,
    run_sealing_stage,
    run_metadata_stage,
    run_morphometrics_stage,
)

## Configuration

Parameters live in `data/processed/config/spine_preprocessing.yaml`; keyword arguments override the file.

In [ ]:
RAW_DATA_ROOT = Path("/path/to/raw/datasets")

cfg = SpinePreprocessingConfig.from_yaml(
    REPO_ROOT / "data" / "processed" / "config" / "spine_preprocessing.yaml",
    raw_data_root=RAW_DATA_ROOT,
    metadata_root=REPO_ROOT / "datasets",
    output_root=None,      # None -> <dataset_root>/preprocessed
    dataset=None,          # e.g. "minnie65" or "h01"
    limit=None,            # small integer for smoke tests
    workers=1,             # >1 uses a process pool
    verbose=False,         # per-spine step timings
)

In [ ]:
records = discover_spines(cfg)
len(records), records[:3]

## Stage 1: seal meshes

In [ ]:
sealed = run_sealing_stage(cfg, records)
sealed["status"].value_counts(dropna=False)

## Stage 2: detect false spines

In [ ]:
false_spines = run_false_spine_detection_stage(cfg, records)
false_spines["status"].value_counts(dropna=False)

## Stage 3: local coordinate frame

In [ ]:
oriented = run_orientation_stage(cfg, records)
oriented["status"].value_counts(dropna=False)

## Stage 5: geometric QC of `local_sealed` meshes

In [ ]:
quality = run_mesh_qc_stage(cfg, records)
quality["status"].value_counts(dropna=False)

## Stage 6: point clouds

In [ ]:
pointclouds = run_pointcloud_stage(cfg, records)
pointclouds["status"].value_counts(dropna=False)

## Stage 7: SDF samples

In [ ]:
sdf = run_sdf_stage(cfg, records)
sdf["status"].value_counts(dropna=False)

## Stage 8: morphometrics

In [ ]:
morphometrics = run_morphometrics_stage(cfg, records)
morphometrics["status"].value_counts(dropna=False)

## Stage 9: metadata

In [ ]:
metadata = run_metadata_stage(cfg, records)
metadata["status"].value_counts(dropna=False)

## Stage 4: branch merge metadata + manifest

In [ ]:
manifest = run_manifest_stage(cfg, records)
manifest[["status", "quality_status", "train_eligible"]].value_counts(dropna=False)

## Full pipeline

In [ ]:
pipeline_result = run_full_spine_preprocessing_pipeline(cfg)
{name: frame["status"].value_counts(dropna=False) for name, frame in pipeline_result.items()}

## Build spines_info.parquet

In [ ]:
info_cfg = SpinesInfoConfig(
    raw_data_root=cfg.raw_data_root,
    processed_root=REPO_ROOT / "data" / "processed" / "neuron-model" / "spines",
    preprocessing_output_root=cfg.output_root,
    metadata_root=cfg.metadata_root,
    dataset=cfg.dataset,
    limit=cfg.limit,
)

spines_info = build_spines_info(info_cfg)
spines_info.head()